# Train a small U-Net for road + building segmentation

PLEM has had no trained extractor until now -- `real_data_evaluation.ipynb` deliberately runs real ground truth through synthetic perturbations as a stand-in "pred" (no GPU available, so training was out of scope there). This notebook closes that gap: it trains a small, CPU-only, single multiclass U-Net (background / road / building) on patches cut from the curated SpaceNet sample built by `spacenet_data_prep.ipynb`, evaluates it on held-out real tiles with PLEM's own metrics (`evaluate_all` -- CBHM, DTAF1, and their area-weighted companions), and renders qualitative image/GT/prediction panels for human review.

**Scope caveat:** this is a first end-to-end pipeline, not a tuned extractor -- the training set is a curated sample of roughly 100 real tiles (not the full SpaceNet corpus), training is CPU-only, and the model is intentionally small (a handful of channels per level). Expect modest quantitative scores; the point is to exercise the full train -> predict -> score -> inspect loop with a real (if weak) model in place of the ground-truth-as-prediction stand-in used elsewhere.

## Colab setup (skipped automatically when running locally)

This notebook is CPU-only by design (see `DEVICE = torch.device("cpu")` below), so it runs fine on a plain Colab CPU runtime -- no GPU needed or used. If you're opening this in Colab: **Runtime > Change runtime type > Hardware accelerator > None**, then **Run all**. The cell below clones this repo into `/content/PLEM`, installs the handful of dependencies Colab's default image doesn't already ship (`torch`/`numpy`/`scipy`/`pandas`/`matplotlib` are preinstalled), and `chdir`s into `notebooks/` so every relative path below (`sys.path.insert(0, os.path.abspath(".."))`, `Path("..").resolve()`) resolves exactly as it does locally. A later cell auto-builds the SpaceNet tile cache (`data/spacenet`) if it's missing, since `data/` is gitignored and each Colab notebook gets its own fresh, empty VM. When run outside Colab this whole section is a no-op.

In [ ]:
import sys, os, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/DanielZGeorge/PLEM.git"
    REPO_DIR = "/content/PLEM"

    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    # torch/numpy/scipy/pandas/matplotlib/requests/pillow ship with the Colab
    # image already -- only install what metrics/datasets additionally need.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "scikit-image", "networkx", "rasterio", "shapely"],
        check=True,
    )

    os.chdir(f"{REPO_DIR}/notebooks")
    print(f"Colab detected -- repo ready at {REPO_DIR}, cwd set to {os.getcwd()}")
else:
    print("Not running in Colab -- skipping repo clone/dependency install.")

In [3]:
import sys, os, random
from pathlib import Path
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from metrics import evaluate_all

# Resolve to an absolute path so this reads/writes the repo's data/ and
# models/ dirs regardless of the kernel's working directory (nbconvert runs
# notebooks with CWD set to the notebook's own folder, not the repo root).
DATA_DIR = Path("..").resolve() / "data"
MODELS_DIR = Path("..").resolve() / "models"
MODELS_DIR.mkdir(exist_ok=True)

SEED = 0
PATCH = 256
DEVICE = torch.device("cpu")  # hardcoded: PLEM's U-Net pipeline is CPU-only by design,
# so this holds even if a GPU is technically available (e.g. a Colab GPU runtime left
# on by mistake) -- switch the Colab runtime to "None" to avoid reserving one for nothing.
if torch.cuda.is_available():
    print("NOTE: CUDA is available but unused -- this notebook always trains on CPU. "
          "Consider Runtime > Change runtime type > Hardware accelerator > None.")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Fixed categorical palette, reused verbatim from real_data_evaluation.ipynb
# so charts read consistently across notebooks -- weighted/lenient companion
# metrics reuse their base metric's color and are distinguished by linestyle.
METRIC_COLORS = {
    "cbhm": "#2a78d6",            # blue
    "cbhm_soft": "#2a78d6",       # blue (dashed)
    "dtaf1": "#4a3aa7",           # violet
    "dtaf1_weighted": "#4a3aa7",  # violet (dashed)
    "cldice_mean": "#1baf7a",     # aqua
    "bf_mean": "#eb6834",         # orange
}
DASHED_METRICS = {"cbhm_soft", "dtaf1_weighted"}


def style_axes(ax):
    ax.grid(True, color="#e1e0d9", linewidth=0.8)
    ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)


def overlay(label):
    """Color-code a label map: road (class 1) red, building (class 2) green."""
    o = np.zeros((*label.shape, 3), dtype=np.uint8)
    o[label == 1] = [255, 0, 0]
    o[label == 2] = [0, 255, 0]
    return o

### Auto-build the SpaceNet cache on Colab if missing

`data/` is gitignored, so a fresh Colab clone has no cached tiles, and each Colab notebook you open gets its own separate runtime/VM -- so running `spacenet_data_prep.ipynb` as a *separate* notebook first won't reliably populate this notebook's `data/spacenet`. Instead, this cell calls the same `build_spacenet_sample` helper that notebook uses, inline, only when running in Colab and only if `data/spacenet` is empty (a no-op on subsequent runs, and a no-op entirely when running locally where `spacenet_data_prep.ipynb` is normally run once ahead of time). Downloads + rasterizes over the network -- expect a few minutes per city the first time.

In [ ]:
if IN_COLAB and (not os.path.isdir(DATA_DIR / "spacenet")
                  or not any((DATA_DIR / "spacenet").iterdir())):
    from datasets.spacenet import build_spacenet_sample

    # Same cities/counts/seed as spacenet_data_prep.ipynb, so the cache this
    # produces is identical to running that notebook ahead of time.
    CITIES = ["Vegas", "Khartoum", "Paris", "Shanghai"]
    N_TILES_PER_CITY = 25
    N_ROAD_CANDIDATES = 100

    print("data/spacenet is empty -- building the curated SpaceNet cache now "
          "(this is what spacenet_data_prep.ipynb does; a few minutes per city "
          "over the network)...")
    for city in CITIES:
        print(f"Building sample for {city}...")
        samples = build_spacenet_sample(
            city, n_tiles=N_TILES_PER_CITY, seed=0,
            cache_dir=str(DATA_DIR / "spacenet"), n_road_candidates=N_ROAD_CANDIDATES,
        )
        print(f"  {len(samples)} tiles cached")
else:
    print("data/spacenet already populated (or running locally) -- skipping SpaceNet download.")

## Load tiles and split train/val/test

Loads every cached SpaceNet `.npz` (run `spacenet_data_prep.ipynb` first) across all cities under `data/spacenet/`. The split happens at the **tile** level, before patchifying -- splitting at the patch level would leak adjacent patches from the same tile across train/val/test and inflate held-out scores.

In [4]:
all_tiles = []
for city_dir in sorted((DATA_DIR / "spacenet").iterdir()):
    if not city_dir.is_dir():
        continue
    for p in sorted(city_dir.glob("*.npz")):
        d = np.load(p)
        all_tiles.append({
            "city": city_dir.name, "tile": p.stem,
            "image": d["image"], "label": d["label"],
        })
print(f"Loaded {len(all_tiles)} SpaceNet tiles across "
      f"{sorted(set(t['city'] for t in all_tiles))}")

rng = np.random.default_rng(SEED)
perm = rng.permutation(len(all_tiles))
n = len(all_tiles)
n_train = int(round(0.70 * n))
n_val = int(round(0.15 * n))

train_tiles = [all_tiles[i] for i in perm[:n_train]]
val_tiles = [all_tiles[i] for i in perm[n_train:n_train + n_val]]
test_tiles = [all_tiles[i] for i in perm[n_train + n_val:]]
print(f"train={len(train_tiles)} tiles  val={len(val_tiles)} tiles  test={len(test_tiles)} tiles")

FileNotFoundError: [Errno 2] No such file or directory: '/content/PLEM/data/spacenet'

## Patchify

Full tiles are 1300x1300px -- too large for a CPU-trained batch. Each tile is reflect-padded (image) / zero-padded (label, i.e. background) up to the next multiple of `PATCH` and cut into a non-overlapping `PATCH`x`PATCH` grid. `stitch_predictions` (used later, at test time) reverses this: it reassembles per-patch predictions into a full padded canvas and crops back to the tile's original shape, so `evaluate_all` always scores a real full-size tile, not an isolated patch.

In [ ]:
def pad_to_multiple(image, label, patch=PATCH):
    h, w = label.shape
    new_h = int(np.ceil(h / patch)) * patch
    new_w = int(np.ceil(w / patch)) * patch
    pad_h, pad_w = new_h - h, new_w - w
    image_p = np.pad(image, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")
    label_p = np.pad(label, ((0, pad_h), (0, pad_w)), mode="constant", constant_values=0)
    return image_p, label_p, (h, w)


def patchify_tile(image, label, patch=PATCH):
    """Returns a list of (image_patch, label_patch) covering the whole tile."""
    image_p, label_p, _ = pad_to_multiple(image, label, patch)
    ph, pw = label_p.shape
    patches = []
    for r in range(0, ph, patch):
        for c in range(0, pw, patch):
            patches.append((image_p[r:r + patch, c:c + patch], label_p[r:r + patch, c:c + patch]))
    return patches


def predict_tile(model, image, orig_label_shape, patch=PATCH, device=DEVICE):
    """Runs the model patch-by-patch over a full tile and stitches an HxW prediction map."""
    dummy_label = np.zeros(orig_label_shape, dtype=np.uint8)
    image_p, _, orig_shape = pad_to_multiple(image, dummy_label, patch)
    ph, pw = image_p.shape[:2]
    canvas = np.zeros((ph, pw), dtype=np.uint8)
    model.eval()
    with torch.no_grad():
        for r in range(0, ph, patch):
            for c in range(0, pw, patch):
                patch_img = image_p[r:r + patch, c:c + patch]
                x = torch.from_numpy(patch_img.transpose(2, 0, 1).astype(np.float32) / 255.0)
                x = x.unsqueeze(0).to(device)
                logits = model(x)
                pred = logits.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
                canvas[r:r + patch, c:c + patch] = pred
    h, w = orig_shape
    return canvas[:h, :w]


train_patches = [p for t in train_tiles for p in patchify_tile(t["image"], t["label"])]
val_patches = [p for t in val_tiles for p in patchify_tile(t["image"], t["label"])]
print(f"train_patches={len(train_patches)}  val_patches={len(val_patches)}  (patch={PATCH}px)")

## Dataset / DataLoader

Training patches get random horizontal/vertical flips (cheap augmentation given how few real tiles are available); validation patches don't.

In [ ]:
class PatchDataset(Dataset):
    def __init__(self, patches, augment=False):
        self.patches = patches
        self.augment = augment

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        image, label = self.patches[idx]
        if self.augment:
            if random.random() < 0.5:
                image, label = image[:, ::-1], label[:, ::-1]
            if random.random() < 0.5:
                image, label = image[::-1, :], label[::-1, :]
        image_t = torch.from_numpy(np.ascontiguousarray(image.transpose(2, 0, 1)).astype(np.float32) / 255.0)
        label_t = torch.from_numpy(np.ascontiguousarray(label).astype(np.int64))
        return image_t, label_t


BATCH_SIZE = 16
train_loader = DataLoader(PatchDataset(train_patches, augment=True), batch_size=BATCH_SIZE,
                           shuffle=True, num_workers=0)
val_loader = DataLoader(PatchDataset(val_patches, augment=False), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=0)

## Small U-Net

A compact 3-level encoder/decoder (base=16 channels, doubling per level) with a single 3-class softmax head (0=background, 1=road, 2=building) -- one multiclass model rather than two binary ones, so its raw `argmax` output is already an `H×W` label map that `evaluate_all()` can score directly, matching PLEM's label convention throughout.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet(nn.Module):
    def __init__(self, in_ch=3, num_classes=3, base=16):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base)
        self.enc2 = DoubleConv(base, base * 2)
        self.enc3 = DoubleConv(base * 2, base * 4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base * 4, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = DoubleConv(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)
        self.out_conv = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)


model = SmallUNet(in_ch=3, num_classes=3, base=16).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"SmallUNet parameter count: {n_params:,}")

## Loss: class-weighted cross-entropy + soft Dice

Background pixels vastly outnumber road/building pixels (see the per-tile percentages printed in `spacenet_data_prep.ipynb`), so plain cross-entropy would let the model collapse to predicting all-background. Class weights are inverse-frequency (computed from the training patches only, clipped to avoid an extreme road/building weight blowing up early training), combined with a soft multiclass Dice term that's insensitive to that same imbalance.

In [ ]:
NUM_CLASSES = 3
class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for _, label in train_patches:
    class_counts += np.bincount(label.ravel(), minlength=NUM_CLASSES)

class_weights = class_counts.sum() / np.maximum(class_counts, 1)
class_weights = class_weights / class_weights.mean()
class_weights = np.clip(class_weights, 0.1, 20.0)
print("class pixel counts:", class_counts)
print("class weights (bg/road/building):", class_weights)

class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
ce_loss_fn = nn.CrossEntropyLoss(weight=class_weights_t)


def dice_loss(logits, target, num_classes=NUM_CLASSES, eps=1e-6):
    probs = torch.softmax(logits, dim=1)
    target_onehot = nn.functional.one_hot(target, num_classes).permute(0, 3, 1, 2).float()
    dims = (0, 2, 3)
    intersection = (probs * target_onehot).sum(dims)
    union = probs.sum(dims) + target_onehot.sum(dims)
    dice_per_class = (2 * intersection + eps) / (union + eps)
    return 1.0 - dice_per_class.mean()


def combined_loss(logits, target):
    return ce_loss_fn(logits, target) + dice_loss(logits, target)

## Training loop

CPU-only, so the epoch budget is deliberately modest. Tracks train/val loss per epoch and checkpoints the model with the best validation loss to `models/unet_road_building.pt` (`models/` is gitignored -- checkpoints are binary and not committed).

In [ ]:
EPOCHS = 15
LR = 1e-3
CHECKPOINT_PATH = MODELS_DIR / "unet_road_building.pt"

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
history = {"epoch": [], "train_loss": [], "val_loss": []}
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum, train_n = 0.0, 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss = combined_loss(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item() * images.size(0)
        train_n += images.size(0)
    train_loss = train_loss_sum / train_n

    model.eval()
    val_loss_sum, val_n = 0.0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = model(images)
            loss = combined_loss(logits, labels)
            val_loss_sum += loss.item() * images.size(0)
            val_n += images.size(0)
    val_loss = val_loss_sum / val_n

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(f"epoch {epoch:2d}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"  -> new best val_loss, checkpoint saved to {CHECKPOINT_PATH}")

# Reload the best checkpoint (not necessarily the last epoch) for evaluation below.
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
model.eval()
print(f"\nBest val_loss={best_val_loss:.4f}; loaded that checkpoint for evaluation.")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history["epoch"], history["train_loss"], marker="o", color="#2a78d6", label="train_loss")
ax.plot(history["epoch"], history["val_loss"], marker="o", color="#eb6834", label="val_loss")
ax.set_xlabel("epoch")
ax.set_ylabel("combined CE + Dice loss")
ax.set_title("Training curve")
ax.legend(frameon=False)
style_axes(ax)
fig.tight_layout()
plt.show()

## Test-set evaluation with PLEM's metrics

Runs inference patch-by-patch on every held-out test tile and stitches predictions back into a full-tile label map via `predict_tile`, then scores each tile with `evaluate_all()` (`metrics/unified.py`) exactly as `real_data_evaluation.ipynb` does for its perturbation sweeps -- so these numbers are directly comparable to that notebook's `cbhm`/`dtaf1` results on the same underlying SpaceNet sample (now scored against a real trained model instead of ground truth run through synthetic perturbations).

In [ ]:
test_rows = []
test_preds = {}  # tile stem -> stitched prediction, kept for the qualitative panels below

for t in test_tiles:
    pred = predict_tile(model, t["image"], t["label"].shape)
    test_preds[t["tile"]] = pred
    result = evaluate_all(pred, t["label"], linear_classes=[1], polygon_classes=[2])
    test_rows.append({
        "city": t["city"], "tile": t["tile"],
        "cbhm": result["cbhm"], "cbhm_soft": result["cbhm_soft"],
        "dtaf1": result["dtaf1"], "dtaf1_weighted": result["dtaf1_weighted"],
        "cldice_mean": result["cldice_mean"], "bf_mean": result["bf_mean"],
    })

test_results = pd.DataFrame(test_rows)
test_results.to_csv(DATA_DIR / "train_unet_test_results.csv", index=False)
test_results

In [ ]:
metric_cols = ["cbhm", "cbhm_soft", "dtaf1", "dtaf1_weighted", "cldice_mean", "bf_mean"]
summary = test_results[metric_cols].agg(["mean", "std"]).T
print(f"Test-set summary across {len(test_results)} held-out tiles:")
display(summary)

fig, ax = plt.subplots(figsize=(7, 4))
means = summary["mean"]
stds = summary["std"]
colors = [METRIC_COLORS[m] for m in metric_cols]
ax.bar(metric_cols, means.values, yerr=stds.values, color=colors, capsize=4)
ax.set_ylim(0, 1.05)
ax.set_ylabel("score")
ax.set_title("Trained U-Net vs. ground truth -- test-set metric means (+/- std)")
style_axes(ax)
fig.tight_layout()
plt.show()

## Qualitative review: image / GT / prediction

For human inspection, not scoring -- three panels per test tile: the RGB image with the GT overlay (red=road, green=building), the same RGB with the model's prediction overlay, and the raw predicted mask on its own so its shape is visible without the underlying imagery. Each tile's `cbhm`/`dtaf1` scores are printed above its row so the qualitative read and the quantitative score can be checked against each other directly.

In [ ]:
N_EXAMPLES = min(4, len(test_tiles))
# Spread across the score range (worst -> best) rather than the first N tiles,
# so review isn't biased toward whichever tiles happen to sort first.
example_order = test_results.sort_values("cbhm")["tile"].tolist()
example_stems = [example_order[i] for i in np.linspace(0, len(example_order) - 1, N_EXAMPLES).round().astype(int)]
tiles_by_stem = {t["tile"]: t for t in test_tiles}

fig, axes = plt.subplots(N_EXAMPLES, 3, figsize=(13.5, 4.5 * N_EXAMPLES))
if N_EXAMPLES == 1:
    axes = axes[None, :]

for row, stem in enumerate(example_stems):
    t = tiles_by_stem[stem]
    gt, pred, image = t["label"], test_preds[stem], t["image"]
    scores = test_results.loc[test_results["tile"] == stem].iloc[0]

    axes[row, 0].imshow(image)
    axes[row, 0].imshow(overlay(gt), alpha=0.5)
    axes[row, 0].set_title(f"{stem} -- GT")
    axes[row, 1].imshow(image)
    axes[row, 1].imshow(overlay(pred), alpha=0.5)
    axes[row, 1].set_title(f"{stem} -- prediction")
    axes[row, 2].imshow(overlay(pred))
    axes[row, 2].set_title("raw prediction mask")
    for col in range(3):
        axes[row, col].axis("off")

    print(f"{stem}:  cbhm={scores['cbhm']:.3f}  cbhm_soft={scores['cbhm_soft']:.3f}  "
          f"dtaf1={scores['dtaf1']:.3f}  dtaf1_weighted={scores['dtaf1_weighted']:.3f}  "
          f"cldice_mean={scores['cldice_mean']:.3f}  bf_mean={scores['bf_mean']:.3f}")

fig.tight_layout()
plt.show()

## Observations

- This is the first real trained-model pipeline in PLEM -- previously all real-data evaluation (`real_data_evaluation.ipynb`) used ground truth run through synthetic perturbations as a stand-in prediction, specifically because no GPU was available for training. This notebook shows that a genuinely small U-Net (3-level encoder/decoder, base=16 channels) trains to convergence on CPU in a reasonable time against a ~100-tile curated SpaceNet sample.
- Expect this model to underperform a production extractor: ~100 tiles (split 70/15/15 train/val/test) is a small dataset for segmentation, and the model has no pretraining. Treat the test-set scores here as a pipeline sanity check (train -> predict -> score -> inspect all work end-to-end against PLEM's own metrics), not as a benchmark result.
- If scores are surprisingly low or the qualitative panels show all-background predictions, check the class-weight printout above (`class weights (bg/road/building)`) -- an extreme weight (near the 0.1/20.0 clip bounds) usually means a training city was skewed toward one class and the loss balance needs revisiting, rather than the model architecture itself being at fault.